# Workshop 3 Lab — Intermediate
### Computation 101 · IQ Biology Bootcamp 2026

You're about to reconstruct a real RNA-seq analysis in Google Colab — a full Linux computer in your
browser. **Run each cell in order** (Shift+Enter). Where you see `...`, replace it with your code before running.

*Background:* **ZFP36L2** is an RNA-binding protein that grabs certain messenger RNAs and marks them for
destruction. Knock the gene out in a mouse (**KO**), and those mRNAs are no longer destroyed — so they go
**up**. Researchers measured this in six tissues with RNA-seq, then used DESeq2 to score every gene. A gene
counts as **up-regulated** when its `log2FoldChange > 1` *and* `padj < 0.05`.

In [ ]:
# One-time setup. The lines starting with ! are REAL shell commands running in this
# Colab Linux machine — the same commands you'd type on Fiji. This clones the course
# repo (only if it isn't here yet) and points DATA at the ZFP36L2 dataset inside it.
![ -d iqbio-computation-101-2026 ] || git clone --depth 1 https://github.com/gsstephenson/iqbio-computation-101-2026.git
import pandas as pd, os
DATA = 'iqbio-computation-101-2026/data/zfp36l2'
print('data folder contains:', os.listdir(DATA))

## Your job: all six tissues, then the overlap
Build the set of up-regulated genes for each of the six tissues, then find the genes that appear
in **all six** at once. Sets make the overlap a one-liner.

In [ ]:
tissues = ['Lung', 'Liver', 'BM', 'Spleen', 'Ovary', 'Kidney']
up = {}
for t in tissues:
    df = pd.read_csv(os.path.join(DATA, 'de', f'{t}_DE.csv'), index_col=0)
    # keep the gene IDs (the index) of the up-regulated genes as a set
    up[t] = set(df[(df['log2FoldChange'] > 1) & (df['padj'] < ...)].index)
    print(f'{t:7s} {len(up[t]):5d} up')

Wildly different counts per tissue. Now the payoff — the **intersection**: which gene IDs are in every set?

In [ ]:
# up is a DICT of sets — set.intersection(*up.values()) keeps only genes present in ALL of them.
universal = ...
print(len(universal), 'gene(s) up in all six tissues:')
print(universal)

**Exactly one gene** survives all six filters: `ENSMUSG00000091694`. Out of ~17,000 genes, one is
up in every tissue when ZFP36L2 is gone. Hold onto that ID — the **Advanced** workbook names it and
asks whether ZFP36L2 physically binds it.

## The volcano plot — the RNA-seq portrait
One dot per gene: fold change across, significance up. Load bone marrow's full table and plot
`log2FoldChange` vs `-log10(padj)`; the genes you've been counting light up in the top-right.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
bm = pd.read_csv(os.path.join(DATA, 'de', 'BM_DE.csv'), index_col=0).dropna(subset=['padj'])
sig = (bm['log2FoldChange'] > 1) & (bm['padj'] < 0.05)
plt.figure(figsize=(6, 5))
plt.scatter(bm['log2FoldChange'], -np.log10(bm[...]), s=4, c='lightgray')     # y-axis = significance
plt.scatter(bm.loc[sig, 'log2FoldChange'], -np.log10(bm.loc[sig, 'padj']), s=4, c='crimson')
plt.xlabel('log2 fold change'); plt.ylabel('-log10(padj)'); plt.title('Bone marrow — volcano'); plt.show()

## See the tissue-selectivity — a heatmap
The paper's headline is *tissue-selective* targeting. `expression_table1.csv` holds the WT expression of the
up-regulated genes across all six tissues. A heatmap makes it jump out: each row a gene, each column a tissue,
color = expression.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
ex = pd.read_csv(os.path.join(DATA, 'expression_table1.csv'), index_col=0)
# the most variable genes across tissues make the clearest picture
top = ex.loc[ex.var(axis=...).sort_values(ascending=False).index[:40]]   # variance ACROSS the tissue columns
plt.figure(figsize=(5, 8))
plt.imshow(np.log1p(top.values), aspect='auto', cmap='magma')
plt.xticks(range(6), top.columns, rotation=45); plt.yticks([])
plt.colorbar(label='log(expression + 1)'); plt.title('Up-genes across six tissues'); plt.show()

## One last lesson: the statistic won't save you
You just drew pictures of your data. Here's *why* that matters — a trick every bioinformatician meets once.
Four tiny datasets. First, the numbers:

In [ ]:
import numpy as np, matplotlib.pyplot as plt
x123 = [10,8,13,9,11,14,6,4,12,7,5]; x4 = [8,8,8,8,8,8,8,19,8,8,8]
quartet = {'I':(x123,[8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]),
           'II':(x123,[9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]),
           'III':(x123,[7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]),
           'IV':(x4,[6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89])}
for k,(x,y) in quartet.items():
    x=np.array(x,float); y=np.array(y,float); m,b=np.polyfit(x,y,1)
    print(f'{k:3}  mean_x={x.mean():.1f}  mean_y={y.mean():.2f}  corr={np.corrcoef(x,y)[0,1]:.3f}  fit: y={m:.2f}x+{b:.2f}')

Identical — same means, the same correlation (0.816), the same best-fit line. By the numbers, one dataset.
Now **look**:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7, 6))
for ax,(k,(x,y)) in zip(axes.ravel(), quartet.items()):
    x=np.array(x,float); y=np.array(y,float); m,b=np.polyfit(x,y,1)
    ax.scatter(x, y, c='crimson'); ax.plot([2,20],[m*2+b, m*20+b], 'k--', lw=0.8)
    ax.set_title(f'Set {k}'); ax.set_xlim(2,20); ax.set_ylim(2,14)
plt.tight_layout(); plt.show()

Same statistics, four different truths: **I** a real linear trend; **II** an obvious curve (a straight line
is the *wrong model*); **III** a clean line wrecked by one **outlier**; **IV** a single high-leverage point
that invents the whole correlation. This is **Anscombe's Quartet** (1973) — and it is the entire reason you
plot. The number never warned you; you had to look.

It's also why RNA-seq lives in **log2 space** and why DESeq2 normalizes by median-of-ratios instead of raw
ratios: choose the right transform, then *validate by eye* with a volcano or MA plot — exactly what you just
did. Don't hedge the statistic. Look at your data, pick the right normalization, and trust the picture.

## One more thing

The dataset you just analyzed isn't a textbook toy. It's a real, published study —
*The RNA binding protein ZFP36L2 displays tissue-selective mRNA targeting in mice*, **RNA Biology (2026)**.

Its first author is **George** — a first-year in your own program, who a year ago sat exactly where
you're sitting now. Everything you just reproduced, a peer generated. That's the whole arc of this
bootcamp: the person one year ahead of you did real science, and today you re-derived their headline
result from their deposited data. Next year, someone reproduces yours.